# TotalSpineSeg on Spine-Generic HEALTHY pilot (native 0.8mm + 4mm control)

Segments 24 cervical T2w (12 healthy subjects × {native 0.8mm, downsampled 4mm}) to get
per-vertebra + **canal** masks for Group 5.2 morphometry. Goal: does healthy Ha/Hp ~0.97
hold at 0.8mm, and does it sag at 4mm? (separates real degeneration from resolution).

1. Runtime → Change runtime type → **T4 GPU**. 2. Put **`spine_generic_pilot.zip`** in My Drive root. 3. Run all.
~24 cases × ~1–3 min on GPU. Resumable (skips done).

### 1 — GPU

In [ ]:
!nvidia-smi -L

### 2 — install TotalSpineSeg (~3–5 min); pin kornia<0.8

In [ ]:
!pip install -q totalspineseg nnunetv2
!pip install -q "kornia<0.8"
import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())

### 3 — mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

### 4 — unzip the pilot bundle

In [ ]:
import os, glob
!unzip -q -o /content/drive/MyDrive/spine_generic_pilot.zip -d /content/sg
os.makedirs('/content/drive/MyDrive/spinegeneric_masks', exist_ok=True)
inputs = sorted(glob.glob('/content/sg/*.nii.gz'))
print(len(inputs), 'images:', [os.path.basename(f) for f in inputs])

### 5 — run TSS FULL mode (no --step1); save step2 (vertebrae+canal) to Drive; resumable

In [ ]:
import os, glob, shutil, time, subprocess
OUT='/content/drive/MyDrive/spinegeneric_masks'; os.makedirs(OUT, exist_ok=True)
inputs=sorted(glob.glob('/content/sg/*.nii.gz'))
print(len(inputs),'cases\n')
for i,f in enumerate(inputs,1):
    base=os.path.basename(f)[:-7]
    dst=f'{OUT}/{base}_step2.nii.gz'
    if os.path.exists(dst): print(f'[{i}/{len(inputs)}] skip {base}'); continue
    outdir=f'/content/out/{base}'; t=time.time(); print(f'[{i}/{len(inputs)}] {base} ...', flush=True)
    r=subprocess.run(['totalspineseg', f, outdir, '--device','cuda'], capture_output=True, text=True)
    seg=glob.glob(f'{outdir}/step2_output/*.nii.gz')
    if r.returncode==0 and seg: shutil.copy(seg[0], dst); print(f'    ok {time.time()-t:.0f}s')
    else: print('    ERROR:', (r.stderr or r.stdout)[-700:])
print('\nDONE. step2 masks on Drive:', len(glob.glob(f'{OUT}/*_step2.nii.gz')))

### Done — download **My Drive/spinegeneric_masks/** `*_step2.nii.gz` into `~/dev/group5-proto/out_sg/`, tell Claude.